# 09 - Documenting Array Dimensions and Signal Metadata

This notebook focuses on good scientific coding practices: documenting array shapes, dtypes, and signal metadata.

In [ ]:
import numpy as np

## Why Documentation Matters

Undocumented axes cause bugs. Consider:
- A `[1024, 2]` array vs a `[2, 1024]` array look similar but have completely different semantics
- Forgetting that axis 0 = samples vs axis 0 = channels leads to axis-swap bugs
- Without metadata (sample rate, units, etc.), data is meaningless

## Template Pattern for Array Documentation

Immediately after loading or transforming an array, write a documentation block:

```
Variable: <name>
Shape: <shape tuple>
Dtype: <dtype>
Dimensions meaning: [dim0 meaning, dim1 meaning, ...]
Sample rate: <fs>
Units: <units>
Notes: <any other relevant info>
```

In [ ]:
# Generate dummy IQ data
np.random.seed(42)
n_samples = 4096
sample_rate = 1e6
center_freq = 915e6

t = np.arange(n_samples) / sample_rate
iq = (1.0 + 0.5j) * np.exp(2j * np.pi * 100e3 * t) + 0.1 * (np.random.randn(n_samples) + 1j * np.random.randn(n_samples))

# DOCUMENTATION BLOCK
# ==================
# Variable: iq
# Shape: (4096,) — 1D complex array
# Dtype: complex128
# Dimensions meaning: [time samples]
# Sample rate: 1 MHz (1e6 samples/sec)
# Center frequency: 915 MHz
# Units: Volts (assumed)
# Notes: Synthetic IQ signal with 100 kHz offset tone + noise
# ==================

print(f"Variable: iq")
print(f"Shape: {iq.shape}")
print(f"Dtype: {iq.dtype}")
print(f"Sample rate: {sample_rate/1e6:.1f} MHz")
print(f"Center frequency: {center_freq/1e6:.0f} MHz")
print(f"Number of samples: {iq.shape[0]}")
print(f"Duration: {iq.shape[0]/sample_rate*1e6:.1f} us")

## Documenting Transformed Arrays

In [ ]:
# Windowed segment
window_start = 512
window_size = 1024
segment = iq[window_start:window_start + window_size]

# DOCUMENTATION BLOCK
# ==================
# Variable: segment
# Shape: (1024,) — 1D complex array
# Dtype: complex128
# Dimensions meaning: [time samples]
# Sample rate: 1 MHz (inherited from iq)
# Time offset: 512 us from start of iq
# Duration: 1024 us (1.024 ms)
# Notes: Windowed segment of iq for detailed analysis
# ==================

print(f"Segment shape: {segment.shape}")
print(f"Time offset: {window_start/sample_rate*1e6:.1f} us")
print(f"Duration: {window_size/sample_rate*1e6:.1f} us")

In [ ]:
# Reshaped into I/Q matrix
iq_matrix = np.column_stack([iq.real, iq.imag])

# DOCUMENTATION BLOCK
# ==================
# Variable: iq_matrix
# Shape: (4096, 2) — 2D float64 array
# Dtype: float64
# Dimensions meaning: [time samples, I/Q components]
#   axis 0 = time (4096 samples)
#   axis 1 = component (0=I, 1=Q)
# Sample rate: 1 MHz
# Units: Volts
# Notes: Complex IQ decomposed into real/imag columns
# ==================

print(f"iq_matrix shape: {iq_matrix.shape}")
print(f"iq_matrix dtype: {iq_matrix.dtype}")
print(f"axis 0: {iq_matrix.shape[0]} time samples")
print(f"axis 1: {iq_matrix.shape[1]} components (I=0, Q=1)")

## Common Undocumented-Axis Bugs

| Bug | Cause | Symptom |
|-----|-------|--------|
| Wrong power values | Transposed [samples, channels] to [channels, samples] | Power mismatch between methods |
| Garbled plots | Swapped I and Q columns | Constellation rotated 90 degrees |
| Wrong FFT results | Applied FFT along wrong axis | Frequency spectrum looks like noise |
| Shape mismatch error | Added arrays with incompatible axis ordering | Broadcasting error |

In [ ]:
# Example: documenting a 3D array
# (e.g., multiple channels over time)
multi_channel = np.random.randn(100, 3, 512)  # [channels, polarization, samples]

# DOCUMENTATION BLOCK
# ==================
# Variable: multi_channel
# Shape: (100, 3, 512) — 3D float64 array
# Dtype: float64
# Dimensions meaning: [channel, polarization, time]
#   axis 0: 100 channels
#   axis 1: 3 polarizations (X, Y, Z)
#   axis 2: 512 time samples per channel
# Sample rate: TBD
# Units: TBD
# ==================

print(f"Shape: {multi_channel.shape}")
print(f"axis 0: {multi_channel.shape[0]} channels")
print(f"axis 1: {multi_channel.shape[1]} polarizations")
print(f"axis 2: {multi_channel.shape[2]} time samples")

## Exercise: Document a New Array

Create a windowed/segmented version of the IQ signal (e.g., split into 4 segments of 1024 samples each stored in a 2D array). Then write a full documentation block for it.

In [ ]:
# YOUR CODE HERE
# Step 1: Create the segmented array
# Step 2: Write the documentation block in a markdown cell below
pass

<details>
<summary>Solution</summary>

```python
# Create 4 segments of 1024 samples
seg_size = 1024
n_segments = 4
segments = iq[:n_segments * seg_size].reshape(n_segments, seg_size)

# Documentation:
# Variable: segments
# Shape: (4, 1024) — 2D complex128 array
# Dimensions meaning: [segment_index, time_samples]
#   axis 0: 4 consecutive time segments
#   axis 1: 1024 time samples per segment
# Sample rate: 1 MHz
# Segment duration: 1024 us each
# Notes: Sequential windowing of iq for time-varying analysis
```
</details>

## Summary

- Always document array shape, dtype, and dimension meaning immediately after creation
- Use a consistent template pattern for documentation
- Undocumented axes are a primary source of bugs, especially axis-swap errors
- Include metadata: sample rate, center frequency, units, and any transformations applied
- This practice ties directly back to the axis-swap debugging exercise in Notebook 08